# Classificação de cobertura do terreno em séries temporais de baixa resolução, com foco em estradas e validação no Brasil

**Classes:** água · vegetação baixa · floresta · construções · **estrada** (+ "outros")
**Eixo central:** quanto o desempenho cai quando a resolução passa de 10 m para 30 m e 64 m, mantendo as mesmas datas.
**Requisito:** todas as métricas finais vêm de **áreas de estudo (AOIs) brasileiras**, separadas geograficamente das áreas de treino.

O notebook roda de ponta a ponta em `MODO_RAPIDO` (padrão), com AOIs recortadas, menos datas e menos épocas. As funções reutilizáveis estão em `src/`, e todos os parâmetros estão em `config.yaml`.

## 1. Introdução e motivação

**Desigualdade de dados.** Territórios como a Amazônia raramente têm a mesma resolução e frequência de imageamento que a Europa e os EUA, onde há mais investimento em mapeamento (ortofotos, LiDAR, constelações comerciais). A nebulosidade constante nos trópicos reduz ainda mais as observações úteis e deixa as séries temporais **irregulares**: o número de datas limpas varia de pixel para pixel e de estação para estação.

**Satélites brasileiros**, com resolução moderada a baixa:
- **CBERS-4A** (Brasil–China, 2019): MUX com 16 m e revisita de 31 dias; WFI com 55 m e revisita de 5 dias; WPM com 2 m (pancromática) e 8 m (multiespectral).
- **Amazonia-1** (2021): primeiro satélite de observação da Terra totalmente brasileiro. A WFI tem faixa de 830 km e cerca de 64 m de resolução, e alimenta o DETER/INPE.

**Programas abertos globais:** **Sentinel-2** (Copernicus/UE), com 10 m e revisita de cerca de 5 dias, e **Landsat** (EUA), com 30 m e longa série histórica. **Planet–NICFI** oferece mosaicos de 4,77 m para os trópicos, incluindo o Brasil. O catálogo do Earth Engine indica disponibilidade até jul/2025. Aqui ele é citado só como apoio para inspeção visual, não como dependência.

**Por que estradas?** Estradas vicinais e não oficiais costumam ser **mais estreitas que um pixel** de 10 a 64 m. Ainda assim, são o vetor principal do desmatamento: o Imazon mapeou cerca de **3,46 milhões de km de estradas na Amazônia Legal em 2020**, a maior parte não oficial (Botelho et al., 2022, *Remote Sensing* 14(15):3625, U-Net + Sentinel-2). Detectar um objeto sub-pixel exige explorar a assinatura espectral misturada e a persistência temporal, e é aí que a resolução pesa mais.

### Tabela comparativa de sensores

| Sensor / produto | Resolução espacial | Revisita | Bandas principais | Acesso | Uso neste estudo |
|---|---|---|---|---|---|
| **Sentinel-2 MSI** (UE) | 10 m (VIS/NIR), 20 m (red-edge/SWIR) | ~5 dias (2 satélites) | 13 bandas, incl. SWIR | Aberto (Copernicus, BDC, Planetary Computer) | Base 10 m; **degradado para 30 e 64 m** |
| **Landsat 8/9 OLI** (EUA) | 30 m (15 m pan) | 16 dias (8 com 2 satélites) | VIS, NIR, SWIR1/2, termal | Aberto (USGS, BDC) | 30 m real (modo completo) |
| **CBERS-4A WFI** (BR–CN) | 55 m (≈64 m nos cubos BDC) | 5 dias | Azul, verde, vermelho, NIR | Aberto (INPE/BDC) | **64 m real**, comparado ao S2 degradado |
| CBERS-4A MUX | 16 m | 31 dias | Azul, verde, vermelho, NIR | Aberto (INPE) | não usado (revisita longa) |
| CBERS-4A WPM | 2 m pan / 8 m MS | 31 dias | Pan + 4 MS | Aberto (INPE) | não usado |
| **Amazonia-1 WFI** (BR) | ~64 m, faixa de 830 km | 5 dias | Azul, verde, vermelho, NIR | Aberto (INPE/BDC) | citado (o BDC ainda não tem cubo regular dele) |
| Planet–NICFI | 4,77 m | mensal/semestral (mosaico) | RGB + NIR | Cadastro NICFI / Earth Engine (até jul/2025) | só inspeção visual (opcional) |

Os cubos do **Brazil Data Cube (INPE)** já entregam composições regulares (8 ou 16 dias, método *Least Cloud Cover First*) com máscara de nuvens. Isso resolve parte da irregularidade temporal, mas não preenche datas sem observação limpa.

## 2. Configuração

`MODO_RAPIDO = True` usa recortes de 9,6 × 9,6 km, 12 datas, poucas épocas e um subconjunto de modelos por experimento, para rodar em CPU. Também dá para trocar o modo pela variável de ambiente `MODO_RAPIDO=0`.

In [ ]:
import os, sys, time, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*POST.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from src.config import load_config, set_seeds, aoi_grid, aois_brasil, RESULTS, DATA, save_json
from src import data as D, labels as L, features as FT, validation as V, metrics as MT, plots as PL
from src.experiments import Store, DATA_KEYS, active_keys, run_experiment, evaluate_predictions, unet_road_experiment

MODO_RAPIDO = None          # None = usa config.yaml / variável de ambiente; True/False força
cfg = load_config(modo_rapido=MODO_RAPIDO)
SEED = cfg["projeto"]["seed"]
set_seeds(SEED)
NOMES = cfg["classes"]["nomes"]
ESTRADA = cfg["classes"]["id"]["estrada"]
T_INICIO = time.time()
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 40)

print("MODO_RAPIDO =", cfg["projeto"]["modo_rapido"], "| ano =", cfg["projeto"]["ano"], "| seed =", SEED)
pd.Series({k: v for k, v in cfg["run"].items() if k != "experimentos"}, name="valor").to_frame()

## 3. Áreas de estudo

As AOIs cobrem biomas e contextos distintos. Todas ficam numa grade UTM com canto alinhado a múltiplos de **960 m** (mmc de 10, 30 e 64), então as grades de 10, 30 e 64 m ficam exatamente aninhadas. `amz_ro_holdout` **nunca entra no treino**, e `eu_alentejo` serve **só para treino** no experimento de transferência.

In [ ]:
aoi_df = pd.DataFrame([{"id": a, "nome": cfg["aois"][a]["nome"], "bioma": cfg["aois"][a]["bioma"],
                        "papel": cfg["aois"][a]["papel"], "pais": cfg["aois"][a]["pais"],
                        "lon": cfg["aois"][a]["centro"][0], "lat": cfg["aois"][a]["centro"][1],
                        "lado_km": cfg["run"]["tamanho_aoi_m"] / 1000, "crs": aoi_grid(cfg, a).crs.to_string(),
                        "justificativa": cfg["aois"][a]["justificativa"]} for a in cfg["run"]["aois"]])
aoi_df

In [ ]:
import folium
cores_papel = {"treino": "#1f77b4", "holdout": "#d62728", "transferencia": "#9467bd"}
mapa = folium.Map(location=[-12, -50], zoom_start=4, tiles="OpenStreetMap")
for a in cfg["run"]["aois"]:
    w, s, e, n = aoi_grid(cfg, a).bounds_lonlat()
    folium.Rectangle([[s, w], [n, e]], color=cores_papel[cfg["aois"][a]["papel"]], weight=3, fill=True, fill_opacity=0.2,
                     tooltip=f"{a}: {cfg['aois'][a]['nome']} ({cfg['aois'][a]['papel']})").add_to(mapa)
    folium.Marker([(s + n) / 2, (w + e) / 2], tooltip=a,
                  icon=folium.Icon(color={"treino": "blue", "holdout": "red", "transferencia": "purple"}[cfg["aois"][a]["papel"]])).add_to(mapa)
mapa

## 4. Aquisição das séries temporais (STAC) e cache local

1. As coleções do **Brazil Data Cube** são listadas em tempo de execução e escolhidas por padrões de texto (`config.yaml → fontes.padroes_colecao`). A escolha fica registrada em `data/escolhas_colecoes.json`.
2. Para cada AOI e cada fonte, só o recorte necessário é lido dos COGs remotos (requisições HTTP *range*) e reprojetado para a grade UTM da AOI. Tudo vai para cache em `data/cubos/`.
3. A AOI europeia (fora do BDC) usa o **Sentinel-2 L2A do Microsoft Planetary Computer**, compondo a mediana das observações limpas em janelas de 16 dias no mesmo calendário do cubo BDC. Se o BDC falhar para uma AOI brasileira, o mesmo caminho serve de alternativa, e isso fica registrado.

Nada é inventado: se uma fonte falha, o erro aparece na tabela de status e a análise segue com o que estiver disponível.

In [ ]:
store = Store(cfg)
colecoes_bdc = D.list_collections(cfg["fontes"]["bdc_stac"])
print(f"{len(colecoes_bdc)} coleções no BDC")
for chave, sensor in [("s2_cubo", "s2"), ("cbers_wfi_cubo", "cbers"), ("landsat_cubo", "landsat")]:
    store.collections[sensor] = D.choose_collection(colecoes_bdc, cfg["fontes"]["padroes_colecao"][chave], chave)
pd.DataFrame(D.load_json(D.CHOICES_FILE, {})).T

In [ ]:
CHAVES = active_keys(cfg)
disponivel = {}
t0 = time.time()
for a in cfg["run"]["aois"]:
    for k in CHAVES:
        if DATA_KEYS[k][0] != "s2" and not store.is_br(a):
            continue                      # sensores do BDC só cobrem o Brasil
        disponivel[(a, k)] = store.available(a, k)
print(f"aquisição/cache: {time.time() - t0:.0f} s")

resumo = []
for (a, k), ok in disponivel.items():
    if ok:
        c = store.cube(a, k)
        resumo.append({"aoi": a, "dados": k, "fonte": str(c["fonte"]), "datas": len(c["dates"]),
                       "bandas": ", ".join(map(str, c["bands"])), "grade": f"{c['valid'].shape[1]}×{c['valid'].shape[2]}",
                       "obs_validas_media": float(c["valid"].sum(0).mean())})
pd.DataFrame(resumo)

**Fontes não usadas por padrão:**
- **Amazonia-1 WFI** (`AMZ1-WFI-L4-SR-1`): o BDC oferece cenas, não um cubo regular. O CBERS-4/4A WFI cumpre o mesmo papel (64 m, 4 bandas) com cubo de 8 dias pronto.
- **Planet–NICFI e Dynamic World**: exigem Earth Engine com login. O caminho opcional está abaixo e **não roda** sem autenticação.

In [ ]:
USAR_EARTH_ENGINE = False   # opcional; requer `earthengine authenticate` (nenhuma credencial neste código)
if USAR_EARTH_ENGINE:
    import ee
    ee.Initialize()
    # MapBiomas no EE: projects/mapbiomas-public/assets/brazil/lulc/v1 (filtrar pelo maior collection_id e pelo ano)
    # Dynamic World: GOOGLE/DYNAMICWORLD/V1 ; NICFI: projects/planet-nicfi/assets/basemaps/americas
    print("Earth Engine inicializado")
else:
    print("Earth Engine desativado: usando GeoTIFF público do MapBiomas, Planetary Computer e BDC.")

## 5. Pré-processamento: nuvens, índices, regularização temporal e degradação de resolução

- **Máscara de nuvens** pelas bandas de qualidade de cada produto:
  - S2 (BDC e PC): `SCL` ∈ {4 vegetação, 5 solo exposto, 6 água, 7 não classificado, 11 neve} = válido.
  - CBERS WFI (BDC): `CMASK` = 127 é válido; 255 indica nuvem/sem dado (verificado pelo histograma abaixo).
  - Landsat (BDC): bits de *fill*, nuvem dilatada, nuvem e sombra do `qa_pixel`.
- **Índices:** NDVI, NDWI (McFeeters), MNDWI e NDBI (estes dois só com SWIR, logo não no CBERS).
- **Regularização:** os cubos BDC já são regulares (16 d no S2 e 8 d no CBERS). No modo rápido, são escolhidas 12 datas igualmente espaçadas no ano. As lacunas **não são preenchidas** para os modelos temporais (que recebem a máscara). Para os atributos agregados, os harmônicos usam interpolação linear no tempo.
- **Degradação:** o S2 de 10 m é agregado por **média das sub-amostras válidas** para 30 m e 64 m. Um pixel grosso só é válido se ≥ 50 % da área for observação limpa.

In [ ]:
# histogram of CBERS CMASK values for one date (documents the clear-value convention)
br = aois_brasil(cfg)
a0 = br[0]
if disponivel.get((a0, "cbers_64")):
    import rasterio
    its = D.search_items(cfg["fontes"]["bdc_stac"], store.collections["cbers"], aoi_grid(cfg, a0, 64).bounds_lonlat(), cfg["projeto"]["ano"])
    cm = D.read_to_grid(its[0].assets["CMASK"].href, aoi_grid(cfg, a0, 64), dtype="float32", src_nodata=-1)
    u, n = np.unique(cm, return_counts=True)
    print(f"CMASK {D.item_date(its[0])} em {a0}:", dict(zip(u.astype(int).tolist(), n.tolist())))

In [ ]:
# irregularity: valid observations per pixel
chaves_plot = [k for k in ["s2_10", "s2_64", "cbers_64"] if k in CHAVES]
fig, axes = plt.subplots(len(chaves_plot), len(br), figsize=(3.2 * len(br), 3 * len(chaves_plot)), squeeze=False)
linhas_obs = []
for i, k in enumerate(chaves_plot):
    for j, a in enumerate(br):
        ax = axes[i, j]
        if not disponivel.get((a, k)):
            ax.set_axis_off(); continue
        c = store.cube(a, k)
        nv = c["valid"].sum(0)
        im = ax.imshow(nv, cmap="viridis", vmin=0, vmax=len(c["dates"]))
        ax.set_title(f"{a}\n{k}: média {nv.mean():.1f}/{len(c['dates'])}", fontsize=8); ax.set_axis_off()
        por_data = c["valid"].mean((1, 2))
        linhas_obs.append({"aoi": a, "dados": k, "datas": len(c["dates"]), "obs_validas_media": nv.mean(),
                           "pct_pixels_<=3_obs": 100 * (nv <= 3).mean(), "data_mais_nublada": c["dates"][por_data.argmin()],
                           "pct_validos_na_pior_data": 100 * por_data.min()})
fig.colorbar(im, ax=axes, shrink=0.6, label="observações válidas")
plt.show()
pd.DataFrame(linhas_obs)

In [ ]:
# same dates, three resolutions + real CBERS 64 m: dry-season median true colour
def rgb(c):
    b = [list(map(str, c["bands"])).index(x) for x in ("red", "green", "blue")]
    d = np.where(c["valid"][:, None], c["data"][:, b].astype(float), np.nan) / 1e4
    img = np.nanmedian(d, 0).transpose(1, 2, 0)
    return np.clip(img / np.nanpercentile(img, 98), 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, k in zip(axes, ["s2_10", "s2_30", "s2_64", "cbers_64"]):
    if disponivel.get((a0, k)):
        ax.imshow(np.nan_to_num(rgb(store.cube(a0, k))), interpolation="nearest")
    ax.set_title(f"{a0} · {k}", fontsize=9); ax.set_axis_off()
plt.suptitle("Mediana anual (cor verdadeira): o mesmo recorte em resoluções diferentes", fontsize=10)
plt.show()

## 6. Construção dos rótulos

### 6.1 Fontes (somente fontes que cobrem o Brasil)

| Classe | Fonte usada no rótulo | Checagem cruzada | Licença | Resolução | Limitações |
|---|---|---|---|---|---|
| Água, floresta, vegetação baixa, construções | **MapBiomas**, coleção mais recente com GeoTIFF público (descoberta listando o bucket) | ESA WorldCover 2021, JRC GSW, MS Building Footprints / OSM | CC-BY 4.0 (MapBiomas, WorldCover, JRC); ODbL (MS, OSM) | 30 m | mapa anual com erros de 5–15 % por classe; "Área Urbanizada" inclui quintais e vias |
| Estrada | **OpenStreetMap** (Overpass via `osmnx`), linhas com buffer por tipo de via | **IBGE BC250** (transporte rodoviário), **DNIT SNV** (rodovias federais); Imazon se acessível | ODbL; IBGE e DNIT são dados públicos | vetorial | OSM tem cobertura **muito desigual** na Amazônia; BC250 está em escala 1:250 000 (erro posicional de dezenas de metros), por isso serve **só para checagem** |
| Europa (transferência) | ESA WorldCover 2021 + OSM | — | CC-BY 4.0 / ODbL | 10 m | ano diferente (2021 vs. 2024) |

O **Amazon Road Dataset / estradas do Imazon** (Botelho et al. 2022) não tem link público de download conhecido (pesquisado em out/2026). O campo `fontes.imazon_estradas` no `config.yaml` está vazio. Se um link for publicado, basta preenchê-lo.

In [ ]:
print("MapBiomas:", D.load_json(D.CHOICES_FILE, {}).get("mapbiomas"))
tabela_mapeamento = L.mapping_table(cfg)
tabela_mapeamento

### 6.2 Estradas como objetos finos: fração de cobertura por pixel

As linhas do OSM recebem um buffer com a largura típica de cada tipo de via (`config.yaml → estradas.largura_m`; vicinais/`track` = 6 m). Os polígonos são rasterizados numa **sub-grade de 2 m** e a média dá a **fração de cobertura** em cada pixel de 10, 30 ou 64 m. É uma área exata, ao contrário do `all_touched`, que marcaria qualquer pixel tocado.

**Limiar documentado e dependente da resolução:** pixel = estrada se `fração ≥ 0,5 × 6 m / resolução` (0,30 a 10 m, 0,10 a 30 m, 0,047 a 64 m), ou seja, metade da cobertura esperada de uma vicinal de 6 m que atravessa o pixel. Um limiar fixo (ex.: 0,15) apagaria todas as vicinais do rótulo a 64 m (6/64 ≈ 0,09), e o estudo passaria a medir o rótulo em vez do sensor. A tabela abaixo mostra a sensibilidade ao fator.

In [ ]:
t0 = time.time()
RES = sorted({DATA_KEYS[k][1] for k in CHAVES})
for a in cfg["run"]["aois"]:
    for r in RES:
        store.labels(a, r)
print(f"rótulos prontos em {time.time() - t0:.0f} s")
sens = pd.concat([L.threshold_sensitivity(store.road_fraction(a, r), cfg, r).assign(aoi=a)
                  for a in br for r in RES])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for r, g in sens.groupby("res_m"):
    gg = g.groupby("fator")["pct_area"].mean()
    ax[0].plot(gg.index, gg.values, "-o", label=f"{r} m")
    gg = g.groupby("fator")["pct_pixels_tocados_mantidos"].mean()
    ax[1].plot(gg.index, gg.values, "-o", label=f"{r} m")
for x in ax: x.axvline(cfg["estradas"]["fator_limiar"], color="k", ls=":", lw=1); x.set_xlabel("fator do limiar"); x.legend(frameon=False); x.grid(alpha=.3)
ax[0].set_ylabel("% da área rotulada como estrada"); ax[1].set_ylabel("% dos pixels tocados por estrada mantidos")
plt.suptitle("Sensibilidade do rótulo de estrada ao limiar (média das AOIs brasileiras)", fontsize=10); plt.show()
sens.pivot_table(index=["res_m", "fator", "limiar"], values=["pct_area", "pct_pixels_tocados_mantidos"], aggfunc="mean")

In [ ]:
fig, axes = plt.subplots(len(RES), len(cfg["run"]["aois"]), figsize=(3 * len(cfg["run"]["aois"]), 3 * len(RES)), squeeze=False)
for i, r in enumerate(RES):
    for j, a in enumerate(cfg["run"]["aois"]):
        PL.show_classes(axes[i, j], store.labels(a, r), NOMES, f"{a} · {r} m")
PL.class_legend(fig, NOMES); plt.subplots_adjust(bottom=0.06); plt.show()

dist = pd.DataFrame({(a, r): pd.Series(np.bincount(store.labels(a, r)[store.labels(a, r) != 255].ravel(), minlength=len(NOMES)),
                                       index=NOMES) / (store.labels(a, r) != 255).sum() * 100
                     for a in cfg["run"]["aois"] for r in RES}).T
dist.index.names = ["aoi", "res_m"]
dist.round(2)

### 6.3 Ruído de rótulo: concordância entre fontes

MapBiomas e OSM são produtos imperfeitos. Para cada AOI brasileira, as checagens abaixo comparam:
- **MapBiomas vs. ESA WorldCover** (ambos mapeados para as classes do estudo, a 30 m): concordância global, kappa e IoU por classe;
- **água:** MapBiomas vs. JRC Global Surface Water (ocorrência ≥ 50 %);
- **construções:** fração de pixels "construções" do MapBiomas que contêm edificações (MS Building Footprints; OSM se não houver arquivo MS para a AOI);
- **estradas:** concordância por comprimento entre OSM, BC250 e DNIT (proporção do comprimento de A a menos de 30 m de B).

In [ ]:
conc = []
for a in br:
    g30 = aoi_grid(cfg, a, 30)
    linha = {"aoi": a}
    mb = store.base_classes(a, 30)
    try:
        wc = D.load_pc_raster(cfg, a, g30, "esa-worldcover", "map", resampling=D.Resampling.mode)
        wc = L.build_lut(cfg["worldcover_para_estudo"], cfg)[np.nan_to_num(wc, nan=0).astype(np.uint8)]
        linha.update({f"MB×WC_{k}": v for k, v in L.class_agreement(mb, wc, NOMES[:5]).items()})
    except Exception as e:
        D.log_status(f"PC:esa-worldcover:{a}", False, e)
    try:
        occ = D.load_pc_raster(cfg, a, g30, "jrc-gsw", "occurrence")
        agua_jrc = np.nan_to_num(occ) >= 50
        agua_mb = mb == cfg["classes"]["id"]["agua"]
        linha["agua_IoU_MB×JRC"] = (agua_jrc & agua_mb).sum() / max((agua_jrc | agua_mb).sum(), 1)
        linha["agua_pct_area_MB"], linha["agua_pct_area_JRC"] = 100 * agua_mb.mean(), 100 * agua_jrc.mean()
    except Exception as e:
        D.log_status(f"PC:jrc-gsw:{a}", False, e)
    try:
        bld, fonte_bld = D.load_buildings(cfg, a, aoi_grid(cfg, a, 10))
        frac_b = L.road_fractions(list(bld.geometry), aoi_grid(cfg, a, 10), [30], 2)[30] if len(bld) else np.zeros(mb.shape)
        urb = mb == cfg["classes"]["id"]["construcoes"]
        linha["edif_fonte"] = fonte_bld
        linha["edif_n"] = len(bld)
        linha["construcoes_MB_com_edificacao_%"] = 100 * (frac_b[urb] > 0).mean() if urb.any() else np.nan
        linha["edificacoes_fora_de_construcoes_MB_%"] = 100 * (urb[frac_b > 0] == 0).mean() if (frac_b > 0).any() else np.nan
    except Exception as e:
        D.log_status(f"Edificacoes:{a}", False, e)
    conc.append(linha)
concordancia_classes = pd.DataFrame(conc).set_index("aoi")
concordancia_classes.T

In [ ]:
fontes_vias = {}
for a in br:
    g10 = aoi_grid(cfg, a, 10)
    fontes_vias[a] = {"OSM": store.roads(a).get("OSM")}
    for nome, fn in [("BC250", D.load_bc250_roads), ("DNIT", D.load_dnit_snv)]:
        try:
            fontes_vias[a][nome] = fn(cfg, a, g10)
        except Exception as e:
            D.log_status(f"{nome}:{a}", False, f"{type(e).__name__}: {e}")
            fontes_vias[a][nome] = None
    if cfg["fontes"]["imazon_estradas"]:
        D.log_status("Imazon", False, "carregador não implementado para o link configurado")
D.log_status("Imazon:estradas", False, "sem link público de download (busca em out/2026); não usado")

linhas = []
for a, fv in fontes_vias.items():
    for x, y in [("OSM", "BC250"), ("OSM", "DNIT"), ("BC250", "DNIT")]:
        r = L.line_agreement(fv.get(x), fv.get(y), tol_m=30)
        linhas.append({"aoi": a, "A": x, "B": y, **r})
concordancia_vias = pd.DataFrame(linhas)
concordancia_vias

**Como ler:** `a_em_b` é a fração do comprimento da fonte A que tem correspondente na fonte B a até 30 m. O BC250 e o DNIT só trazem rodovias oficiais, então `BC250_em_OSM` alto e `OSM_em_BC250` baixo indicam que o OSM tem as oficiais **e** muitas vicinais. Se `OSM_em_BC250` estiver perto de 1, o OSM está incompleto e as vias não oficiais estão faltando no rótulo.

### 6.4 Pontos de validação oficiais do MapBiomas

O MapBiomas publica cerca de 85 mil pontos de validação interpretados visualmente (plataforma TVI/LAPIG, 3 intérpretes por ponto). O arquivo baixado vai até **2022**, então a comparação usa o último ano disponível. Em AOIs de 9,6 km há poucos pontos (≈ 1 a cada 100 km²), e por isso eles servem apenas como checagem adicional, não como métrica principal.

In [ ]:
try:
    pts = D.load_mapbiomas_validation_points(cfg)
    ano_pts = max(int(c[6:]) for c in pts.columns if c.startswith("CLASS_") and c[6:].isdigit() and int(c[6:]) <= cfg["projeto"]["ano"])
    lut = L.build_lut(cfg["mapbiomas_para_estudo"], cfg)
    linhas = []
    for a in br:
        g = aoi_grid(cfg, a, 30)
        w, s, e, n = g.bounds_lonlat()
        sub = pts.cx[w:e, s:n].to_crs(g.crs)
        for _, p in sub.iterrows():
            col = int((p.geometry.x - g.minx) // 30); row = int((g.maxy - p.geometry.y) // 30)
            if 0 <= row < g.shape[0] and 0 <= col < g.shape[1]:
                linhas.append({"aoi": a, "linha30": row, "coluna30": col,
                               "ref_pontos": NOMES[lut[int(p[f"CLASS_{ano_pts}"])]] if lut[int(p[f"CLASS_{ano_pts}"])] != 255 else None,
                               "mapbiomas_mapa": NOMES[store.base_classes(a, 30)[row, col]] if store.base_classes(a, 30)[row, col] != 255 else None})
    pontos_mb = pd.DataFrame(linhas)
    print(f"ano dos pontos: {ano_pts}; pontos dentro das AOIs: {len(pontos_mb)}")
    if len(pontos_mb):
        pontos_mb["concorda"] = pontos_mb.ref_pontos == pontos_mb.mapbiomas_mapa
        display(pontos_mb.groupby("aoi").agg(n=("concorda", "size"), concordancia=("concorda", "mean")))
except Exception as e:
    D.log_status("MapBiomas:pontos_validacao", False, e)
    pontos_mb = pd.DataFrame()
    print("pontos de validação indisponíveis:", e)

### 6.5 Conjunto "ouro" para interpretação manual

A amostragem é aleatória e **estratificada por classe e por AOI brasileira**, na grade de 10 m. O arquivo `validacao/pontos_para_interpretacao.csv` (e `.gpkg`) traz as colunas `rotulo_manual`, `confianca`, `interprete` e `obs` para revisão visual no QGIS (ex.: sobre mosaicos NICFI, Google Satellite ou Bing). O arquivo **nunca é sobrescrito**. Se `rotulo_manual` estiver preenchido, a seção 10 recalcula as métricas usando esses pontos como **referência principal**.

In [ ]:
n_ouro = cfg["validacao_ouro"]["pontos_por_classe_aoi_rapido" if cfg["projeto"]["modo_rapido"] else "pontos_por_classe_aoi_completo"]
amostra = L.stratified_sample({a: store.labels(a, 10) for a in br}, {a: aoi_grid(cfg, a, 10) for a in br}, n_ouro, cfg, SEED)
arq, novo = L.export_gold(amostra, cfg)
print(("exportado" if novo else "já existia (preservado)"), "→", arq)
OURO = L.load_gold(cfg)
print("pontos com rótulo manual:", 0 if OURO is None else len(OURO))
amostra.groupby(["aoi", "classe_automatica"]).size().unstack(fill_value=0)

## 7. Análise exploratória

Curvas temporais médias por classe (com faixa interquartil) e separabilidade entre classes. A separabilidade usa a **distância de Jeffries–Matusita** (JM, 0 a 2; acima de 1,9 = boa separação), calculada sobre atributos agregados, a 10 m e a 64 m. A pergunta é quanto a estrada se mistura com a vegetação baixa quando o pixel cresce.

In [ ]:
def curvas(a, k, var="ndvi", n=4000):
    c = store.cube(a, k); lab = store.labels(a, DATA_KEYS[k][1])
    names = FT.variable_names(c["bands"]); doy = FT.day_of_year(c["dates"])
    out = {}
    for ci, cn in enumerate(NOMES):
        r, cc = np.nonzero(lab == ci)
        if len(r) < 20: continue
        sel = np.random.default_rng(SEED).choice(len(r), min(n, len(r)), replace=False)
        X = FT.series_for_pixels(c, r[sel], cc[sel])[:, :, names.index(var)]
        out[cn] = (doy, np.nanpercentile(X, 25, 0), np.nanmedian(X, 0), np.nanpercentile(X, 75, 0))
    return out

for var in ["ndvi", "mndwi"]:
    fig, axes = plt.subplots(1, len(br), figsize=(4 * len(br), 3.2), sharey=True, squeeze=False)
    for ax, a in zip(axes[0], br):
        for cn, (doy, q1, md_, q3) in curvas(a, "s2_10", var).items():
            ax.plot(doy, md_, "-o", ms=3, color=PL.CLASS_COLORS[cn], label=PL.CLASS_LABELS[cn])
            ax.fill_between(doy, q1, q3, color=PL.CLASS_COLORS[cn], alpha=0.15)
        ax.set_title(a, fontsize=9); ax.set_xlabel("dia do ano"); ax.grid(alpha=.3)
    axes[0, 0].set_ylabel(f"{var.upper()} (S2 10 m)"); axes[0, -1].legend(fontsize=7, frameon=False)
    plt.show()

In [ ]:
def jm(F, y, a, b):
    xa, xb = F[y == a], F[y == b]
    if len(xa) < 30 or len(xb) < 30: return np.nan
    m1, m2 = xa.mean(0), xb.mean(0)
    s1, s2 = np.cov(xa, rowvar=False) + 1e-6 * np.eye(F.shape[1]), np.cov(xb, rowvar=False) + 1e-6 * np.eye(F.shape[1])
    s = (s1 + s2) / 2; d = m1 - m2
    bh = d @ np.linalg.solve(s, d) / 8 + 0.5 * (np.linalg.slogdet(s)[1] - 0.5 * (np.linalg.slogdet(s1)[1] + np.linalg.slogdet(s2)[1]))
    return 2 * (1 - np.exp(-bh))

sel_feats = ["ndvi_media", "ndvi_p10", "ndvi_p90", "ndwi_media", "red_p50", "nir_p50", "blue_p50", "green_p50"]
linhas = []
for k in [x for x in ["s2_10", "s2_30", "s2_64", "cbers_64"] if x in CHAVES]:
    for a in br:
        if not disponivel.get((a, k)): continue
        Fm = store.features(a, k); fn = store.feature_names(a, k)
        cols = [fn.index(f) for f in sel_feats if f in fn]
        lab = store.labels(a, DATA_KEYS[k][1]).ravel()
        rng = np.random.default_rng(SEED); idx = rng.choice(len(lab), min(60000, len(lab)), replace=False)
        F, y = np.asarray(Fm[idx][:, cols]), lab[idx]
        for c2 in ["vegetacao_baixa", "floresta", "construcoes", "outros"]:
            linhas.append({"dados": k, "aoi": a, "par": f"estrada × {c2}", "JM": jm(F, y, ESTRADA, NOMES.index(c2))})
sep = pd.DataFrame(linhas)
sep.pivot_table(index="par", columns="dados", values="JM", aggfunc="mean")[[k for k in ["s2_10", "s2_30", "s2_64", "cbers_64"] if k in sep.dados.unique()]]

## 8. Desenho da validação

1. **Blocos espaciais com margem** (dentro de cada AOI de treino): a AOI é dividida em blocos de `bloco_m` metros, distribuídos aleatoriamente em *k* folds. No fold *f*, o teste são os blocos *f*, e o treino exclui qualquer pixel a menos de `buffer_m` de um pixel de teste. Assim, pixels vizinhos nunca ficam um no treino e outro no teste. Como os blocos são definidos em **metros**, a mesma partição vale para 10, 30 e 64 m.
2. **Holdout geográfico:** `amz_ro_holdout` (Rondônia) nunca é vista no treino.
3. **Leave-one-region-out (LORO):** treina-se em todas as AOIs de treino menos uma e testa-se na que ficou de fora, o que mede a generalização entre biomas.
4. **Transferência Europa → Brasil:** treina-se só em `eu_alentejo` e testa-se em todas as AOIs brasileiras. **O teste é sempre no Brasil.**

As asserções em `validation.check_design` (e em `tests/test_vazamento.py`) garantem que nenhuma AOI de teste é estrangeira e que o holdout não aparece em nenhum treino.

In [ ]:
EXPS = V.design_experiments(cfg)
pd.DataFrame([{"experimento": e["nome"], "tipo": e["tipo"], "treino": ", ".join(e["treino"]), "teste": ", ".join(e["teste"])} for e in EXPS])

In [ ]:
folds = V.assign_block_folds(cfg["run"]["tamanho_aoi_m"], cfg["run"]["bloco_m"], cfg["run"]["folds_blocos"], SEED)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, r in zip(axes, [10, 30, 64]):
    shape = aoi_grid(cfg, br[0], r).shape
    tr, te = V.fold_masks(shape, r, cfg["run"]["bloco_m"], cfg["run"]["buffer_m"], folds, 0)
    vis = np.zeros(shape); vis[tr] = 1; vis[te] = 2
    ax.imshow(vis, cmap=plt.matplotlib.colors.ListedColormap(["#f2c14e", "#4c78a8", "#e45756"]), vmin=0, vmax=2, interpolation="nearest")
    rep = V.leakage_report(tr, te, r, cfg["run"]["buffer_m"])
    ax.set_title(f"{r} m · fold 0\ncompartilhados={rep['pixels_compartilhados']}, treino no buffer={rep['treino_dentro_do_buffer']}", fontsize=9)
    ax.set_axis_off()
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color="#4c78a8", label="treino"), Patch(color="#e45756", label="teste"), Patch(color="#f2c14e", label="margem descartada")],
           loc="lower center", ncol=3, frameon=False)
plt.suptitle(f"Blocos de {cfg['run']['bloco_m']} m com margem de {cfg['run']['buffer_m']} m (mesma partição em todas as resoluções)", fontsize=10)
plt.show()

vaz = []
for r in [10, 30, 64]:
    shape = aoi_grid(cfg, br[0], r).shape
    for f in range(cfg["run"]["folds_blocos"]):
        tr, te = V.fold_masks(shape, r, cfg["run"]["bloco_m"], cfg["run"]["buffer_m"], folds, f)
        vaz.append({"res_m": r, "fold": f, **V.leakage_report(tr, te, r, cfg["run"]["buffer_m"])})
vaz = pd.DataFrame(vaz)
assert (vaz.pixels_compartilhados == 0).all() and (vaz.treino_dentro_do_buffer == 0).all()
vaz

## 9. Treinamento dos modelos

A complexidade cresce aos poucos:
1. **Random Forest** e **LightGBM** sobre atributos temporais agregados (estatísticas e harmônicos de Fourier), com `class_weight="balanced"`.
2. **TempCNN** (CNN 1D) e **L-TAE leve** (atenção temporal com consulta-mestra e codificação por dia do ano), em PyTorch. Recebem a série com **máscara de observações inválidas**, sem preencher lacunas.
3. **U-Net pequena** só para estradas, sobre composições temporais (p10/p50/p90). Roda apenas no modo completo (`usar_unet`).

**Desbalanceamento:** a estrada é rara. Usamos (i) amostragem de treino com teto por classe, que mantém todas as estradas até o teto; (ii) pesos de classe balanceados nas árvores; (iii) **focal loss** (γ = 2) com α ∝ 1/√frequência nas redes. A focal loss foi escolhida porque reduz o peso dos pixels fáceis (floresta homogênea) sem inflar demais a classe rara, o que tende a gerar falsos positivos ao longo de bordas.

No modo rápido, cada experimento roda só os modelos listados em `config.yaml → modos.rapido.experimentos`.

In [ ]:
PRED = {}        # (tipo, modelo, dados) -> {aoi: mapa}
TEMPOS = []
modelos_por_tipo = cfg["run"]["experimentos"]
t0 = time.time()
for e in EXPS:
    for m in modelos_por_tipo[e["tipo"]]:
        for k in CHAVES:
            if not all(disponivel.get((a, k)) for a in e["treino"] + e["teste"]):
                continue                    # e.g. CBERS has no European data -> no transfer run for cbers_64
            out = PRED.setdefault((e["tipo"], m, k), {})
            r = run_experiment(store, e, m, k, out)
            TEMPOS.append({"experimento": e["nome"], "modelo": m, "dados": k, "t_ajuste_s": r["t_ajuste"]})
print(f"treino + predição: {(time.time() - t0) / 60:.1f} min")
pd.DataFrame(TEMPOS).groupby(["modelo", "dados"]).t_ajuste_s.agg(["count", "mean"]).round(1)

In [ ]:
UNET = {}
if cfg["run"]["usar_unet"]:
    treino_br = [a for a in br if cfg["aois"][a]["papel"] == "treino"]
    holdout = [a for a in br if cfg["aois"][a]["papel"] == "holdout"]
    for k in [x for x in CHAVES if DATA_KEYS[x][0] == "s2"]:
        UNET[k] = unet_road_experiment(store, k, treino_br, holdout)
else:
    print("U-Net desativada no modo rápido (usar_unet: false).")

## 10. Avaliação no Brasil

Todas as métricas abaixo vêm de **AOIs brasileiras**. Os intervalos de confiança de 95 % usam **bootstrap por blocos espaciais**: reamostrar pixels autocorrelacionados daria intervalos estreitos demais.

Métricas por classe: precisão, revocação, F1, IoU e matrizes de confusão. Para **estradas**, há ainda:
- **precisão/revocação relaxadas** com tolerância de 1 e 2 pixels;
- **completude e correção por comprimento**: esqueletização das redes de referência e predita, com comprimento medido sobre o esqueleto (o grafo de pixels é vetorizado em `metrics.skeleton_to_lines`), na lógica de Wiedemann et al. usada na avaliação do Imazon.

**Atenção à comparação entre resoluções:** o rótulo é refeito em cada resolução (maioria do MapBiomas + fração de estrada). Por isso o F1 por pixel compara tarefas um pouco diferentes. As métricas **por comprimento**, com tolerância, são as mais comparáveis entre resoluções.

In [ ]:
linhas = []
for (tipo, m, k), mapas in PRED.items():
    linhas += evaluate_predictions(store, mapas, tipo, m, k)
MET = pd.DataFrame(linhas)
CMS = MET.pop("_cm")
assert set(MET.aoi) <= set(br), "há métrica fora do Brasil!"
MET.to_csv(RESULTS / "metricas.csv", index=False)
print("métricas de", MET.aoi.nunique(), "AOIs brasileiras:", sorted(MET.aoi.unique()), "→ resultados/metricas.csv")

def tabela(df, cols):
    out = df.copy()
    for c in cols:
        if f"{c}_ic_inf" in out:
            out[c] = out.apply(lambda r: f"{r[c]:.3f} [{r[c + '_ic_inf']:.2f}, {r[c + '_ic_sup']:.2f}]", axis=1)
    return out

cols = ["f1_macro", "acuracia_global", "f1_estrada", "iou_estrada"]
show = ["experimento", "aoi", "modelo", "dados", "res_m"] + cols + ["f1_relax2", "completude", "correcao"]
tabela(MET[MET.experimento == "holdout"], cols)[show].sort_values(["modelo", "res_m", "dados"])

In [ ]:
# blocks and LORO: average across folds/AOIs (blocks: out-of-fold map per AOI)
agg = (MET.groupby(["experimento", "modelo", "dados", "res_m"])[["f1_macro", "acuracia_global", "f1_estrada", "f1_relax2", "completude", "correcao",
                                                                  "f1_agua", "f1_vegetacao_baixa", "f1_floresta", "f1_construcoes"]]
       .mean().reset_index())
agg.sort_values(["experimento", "modelo", "res_m"])

### Gráfico-chave: F1 de estrada vs. resolução espacial

Cada curva é um modelo, usando o Sentinel-2 nativo a 10 m e o mesmo S2 **degradado** para 30 m e 64 m (mesmas datas). Os losangos mostram o **CBERS-4/4A WFI a 64 m real**. No holdout (uma AOI), a faixa sombreada é o IC de 95 % (bootstrap por blocos). No LORO e nos blocos (várias AOIs), a linha é a média e a faixa vai do mínimo ao máximo entre as AOIs.

In [ ]:
tipos = [t for t in ["holdout", "loro", "blocos"] if t in MET.experimento.unique()]
fig, axes = plt.subplots(2, len(tipos), figsize=(5.2 * len(tipos), 8), squeeze=False)
for j, t in enumerate(tipos):
    d = MET[MET.experimento == t]
    PL.f1_vs_resolution(axes[0, j], d, "f1_estrada", f"{t}: F1 estrada (por pixel)")
    PL.f1_vs_resolution(axes[1, j], d, "f1_relax2", f"{t}: F1 estrada relaxado (2 px)")
plt.tight_layout(); plt.savefig(RESULTS / "f1_estrada_vs_resolucao.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
d = MET[MET.experimento == "holdout"]
PL.f1_vs_resolution(axes[0], d, "completude", "holdout: completude por comprimento")
PL.f1_vs_resolution(axes[1], d, "correcao", "holdout: correção por comprimento")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
PL.f1_vs_resolution(axes[0], d, "f1_macro", "holdout: F1-macro (6 classes)")
PL.f1_vs_resolution(axes[1], d, "acuracia_global", "holdout: acurácia global")
plt.tight_layout(); plt.show()

In [ ]:
# confusion matrices on the holdout AOI (best model by F1-macro at 10 m)
hold = MET[(MET.experimento == "holdout")]
melhor = hold[hold.dados == "s2_10"].sort_values("f1_macro").iloc[-1].modelo
sel = [(i, r) for i, r in hold.iterrows() if r.modelo == melhor]
fig, axes = plt.subplots(2, len(sel), figsize=(3.6 * len(sel), 7), squeeze=False)
for j, (i, r) in enumerate(sel):
    cm = CMS[i]
    for row, (mat, fmt, tit) in enumerate([(cm, "d", "absoluta"), (cm / np.maximum(cm.sum(1, keepdims=True), 1), ".2f", "normalizada (linha)")]):
        ax = axes[row, j]
        ax.imshow(mat, cmap="Blues")
        for (y, x), v in np.ndenumerate(mat):
            ax.text(x, y, format(v, fmt), ha="center", va="center", fontsize=6)
        ax.set_xticks(range(len(NOMES))); ax.set_yticks(range(len(NOMES)))
        ax.set_xticklabels([PL.CLASS_LABELS[n][:6] for n in NOMES], fontsize=7, rotation=45)
        ax.set_yticklabels([PL.CLASS_LABELS[n][:6] for n in NOMES], fontsize=7)
        ax.set_title(f"{melhor} · {r.dados} · {tit}", fontsize=8)
axes[0, 0].set_ylabel("referência"); axes[1, 0].set_xlabel("predito")
plt.tight_layout(); plt.show()

### Mapas de predição e de erro por AOI

In [ ]:
def painel(tipo, m, aois):
    ks = [k for k in CHAVES if (tipo, m, k) in PRED]
    for a in aois:
        ks_a = [k for k in ks if a in PRED[(tipo, m, k)]]
        if not ks_a: continue
        fig, axes = plt.subplots(2, len(ks_a) + 1, figsize=(3 * (len(ks_a) + 1), 6), squeeze=False)
        PL.show_classes(axes[0, 0], store.labels(a, 10), NOMES, f"{a}: referência 10 m")
        axes[1, 0].imshow(np.nan_to_num(rgb(store.cube(a, "s2_10")))); axes[1, 0].set_axis_off(); axes[1, 0].set_title("S2 mediana", fontsize=9)
        for j, k in enumerate(ks_a, 1):
            p = PRED[(tipo, m, k)][a]
            PL.show_classes(axes[0, j], p, NOMES, f"{m} · {k}")
            err = PL.error_map(store.labels(a, DATA_KEYS[k][1]), p)
            axes[1, j].imshow(np.ma.masked_equal(err, 255), cmap=plt.matplotlib.colors.ListedColormap(["#e8e8e8", "#d62728"]), vmin=0, vmax=1, interpolation="nearest")
            axes[1, j].set_title(f"erro ({100 * (err == 1).sum() / max((err != 255).sum(), 1):.0f} %)", fontsize=9); axes[1, j].set_axis_off()
        PL.class_legend(fig, NOMES); plt.subplots_adjust(bottom=0.07); plt.show()

holdout_aois = [a for a in br if cfg["aois"][a]["papel"] == "holdout"]
painel("holdout", melhor, holdout_aois)
m_loro = cfg["run"]["experimentos"]["loro"][-1]
painel("loro", m_loro, [a for a in br if cfg["aois"][a]["papel"] == "treino"])

In [ ]:
# predicted road network (vectorized skeleton) on the holdout AOI, exported for QGIS
import geopandas as gpd
from skimage.morphology import skeletonize
vet = []
for k in CHAVES:
    if ("holdout", melhor, k) not in PRED: continue
    for a, p in PRED[("holdout", melhor, k)].items():
        g = aoi_grid(cfg, a, DATA_KEYS[k][1])
        linhas_v = MT.skeleton_to_lines(skeletonize(p == ESTRADA), g.transform)
        vet.append(gpd.GeoDataFrame({"aoi": a, "dados": k, "modelo": melhor}, index=range(len(linhas_v)), geometry=linhas_v, crs=g.crs).to_crs(4326))
if vet:
    rede = pd.concat(vet, ignore_index=True)
    rede.to_file(RESULTS / "rede_estradas_predita_holdout.gpkg", driver="GPKG")
    print(rede.to_crs(aoi_grid(cfg, holdout_aois[0]).crs).assign(km=lambda d: d.length / 1e3).groupby("dados").km.sum().round(1).rename("km de estrada predita (vetorizada)"))

In [ ]:
# optional U-Net (roads only), compared on the holdout AOI
if UNET:
    linhas = []
    for k, mapas in UNET.items():
        for a, p in mapas.items():
            r = DATA_KEYS[k][1]; lab = store.labels(a, r); m = lab != 255
            gt, pr = (lab == ESTRADA) & m, (p == ESTRADA) & m
            tp = (gt & pr).sum(); prec = tp / max(pr.sum(), 1); rec = tp / max(gt.sum(), 1)
            linhas.append({"modelo": "unet", "dados": k, "res_m": r, "aoi": a, "sensor": "s2",
                           "f1_estrada": 2 * prec * rec / max(prec + rec, 1e-9), **MT.relaxed_pr(gt, pr, m, 2), **MT.length_metrics(gt, pr, m, r)})
    unet_df = pd.DataFrame(linhas)
    display(unet_df)
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    PL.f1_vs_resolution(ax, pd.concat([MET[MET.experimento == "holdout"], unet_df]), "f1_estrada", "holdout: F1 estrada (+ U-Net)", show_cbers=False)
    plt.show()

### Avaliação com o conjunto ouro (se preenchido)

Se `validacao/pontos_para_interpretacao.(csv|gpkg)` tiver a coluna `rotulo_manual` preenchida, as predições do holdout e do LORO são avaliadas nesses pontos. Esses números passam a ser a **referência principal**, e os do MapBiomas/OSM viram secundários.

In [ ]:
OURO = L.load_gold(cfg)
if OURO is None:
    print("Conjunto ouro ainda não interpretado: as métricas acima usam MapBiomas + OSM como referência.")
else:
    linhas = []
    for (tipo, m, k), mapas in PRED.items():
        if tipo not in ("holdout", "loro", "transferencia"): continue
        r = DATA_KEYS[k][1]
        sub = OURO[OURO.aoi.isin(mapas)].copy()
        if sub.empty: continue
        sub["pred"] = [NOMES[mapas[a][int(l) * 10 // r, int(c) * 10 // r]] if mapas[a][int(l) * 10 // r, int(c) * 10 // r] != 255 else None
                       for a, l, c in zip(sub.aoi, sub.linha, sub.coluna)]
        s = MT.per_point_scores(sub, "pred", "rotulo_manual", NOMES)
        s_auto = MT.per_point_scores(sub, "classe_automatica", "rotulo_manual", NOMES)
        linhas.append({"experimento": tipo, "modelo": m, "dados": k, "n_pontos": len(sub), "f1_macro_ouro": s["f1_macro"],
                       "f1_estrada_ouro": s["f1_estrada"], "acuracia_ouro": s["acuracia_global"],
                       "acuracia_rotulo_automatico_vs_ouro": s_auto["acuracia_global"]})
    MET_OURO = pd.DataFrame(linhas)
    MET_OURO.to_csv(RESULTS / "metricas_conjunto_ouro.csv", index=False)
    display(MET_OURO)

## 11. Experimento de transferência Europa → Brasil

O mesmo modelo é treinado com dados europeus (Alentejo, rótulos de WorldCover + OSM, Sentinel-2 do Planetary Computer) e testado nas AOIs brasileiras. A comparação é feita **na mesma AOI de teste**:
- no holdout (RO), contra o modelo treinado nas AOIs brasileiras de treino;
- em cada AOI de treino brasileira, contra o modelo LORO (treinado nas outras AOIs brasileiras).

**Ressalvas:** há *domain shift* de processamento (compósito próprio de mediana do PC vs. LCF do BDC), de legenda (WorldCover vs. MapBiomas) e de ano (WorldCover 2021). Esses fatores fazem parte do argumento: rótulos "baratos" e abundantes de outra região não substituem rótulos locais.

In [ ]:
if "transferencia" in MET.experimento.unique():
    tr = MET[MET.experimento == "transferencia"]
    br_ref = MET[MET.experimento.isin(["holdout", "loro"])]
    comp = tr.merge(br_ref, on=["modelo", "dados", "aoi"], suffixes=("_EU", "_BR"))
    comp = comp[["aoi", "modelo", "dados", "f1_macro_BR", "f1_macro_EU", "f1_estrada_BR", "f1_estrada_EU", "f1_relax2_BR", "f1_relax2_EU"]]
    comp["queda_f1_macro"] = comp.f1_macro_BR - comp.f1_macro_EU
    display(comp.sort_values(["modelo", "dados", "aoi"]))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, met in zip(axes, ["f1_macro", "f1_estrada"]):
        g = comp.groupby(["dados"])[[f"{met}_BR", f"{met}_EU"]].mean().reindex([k for k in CHAVES if k in comp.dados.unique()])
        g.plot.bar(ax=ax, color=["#4c78a8", "#e45756"], rot=0)
        ax.set_title(f"{met}: treino no Brasil vs. na Europa (teste no Brasil)", fontsize=9); ax.legend(["treino BR", "treino EU"], frameon=False); ax.grid(alpha=.3, axis="y")
    plt.tight_layout(); plt.show()
else:
    print("transferência não executada (AOI europeia indisponível)")

## 12. Discussão

O resumo abaixo é calculado a partir das métricas desta execução. A interpretação que segue é geral e deve ser lida junto com as tabelas.

In [ ]:
h = MET[MET.experimento == "holdout"]
res_tab = h.pivot_table(index="modelo", columns="dados", values="f1_estrada").round(3)
print("F1 de estrada no holdout (RO), por modelo × dados:"); display(res_tab)
print("F1-macro no holdout:"); display(h.pivot_table(index="modelo", columns="dados", values="f1_macro").round(3))
if "s2_10" in res_tab and "s2_64" in res_tab:
    q = (res_tab["s2_10"] - res_tab["s2_64"]).mean()
    print(f"Queda média de F1 de estrada de 10 m para 64 m (S2 degradado): {q:+.3f}")
if "s2_64" in res_tab and "cbers_64" in res_tab:
    print(f"Diferença média CBERS 64 m real − S2 degradado 64 m: {(res_tab['cbers_64'] - res_tab['s2_64']).mean():+.3f}")

**Limitações**
- **Ruído de rótulo.** O MapBiomas é um mapa modelado (30 m), com erros conhecidos por classe e por bioma. A 10 m, o rótulo é só o MapBiomas reamostrado (vizinho mais próximo), então as bordas ficam "em degraus" e penalizam modelos que acertam a geometria fina. A classe "construções" (Área Urbanizada) inclui ruas, quintais e árvores urbanas.
- **OSM incompleto.** Na Amazônia, o OSM cobre bem as rodovias oficiais e de forma irregular as vicinais e os ramais. Estradas reais ausentes do OSM viram falsos positivos do modelo, o que **subestima a precisão** de estrada. A concordância OSM × BC250 × DNIT (seção 6.3) e o conjunto ouro existem para medir isso.
- **Nuvens.** Mesmo com compósitos de 8 e 16 dias, há datas com pouquíssimos pixels válidos na estação chuvosa (seção 5). A máscara do L-TAE e da TempCNN lida com a irregularidade, mas não recupera informação ausente.
- **Estradas sub-pixel.** A 64 m, uma vicinal de 6 m ocupa menos de 10 % do pixel. O rótulo depende fortemente do limiar (seção 6.2), e o F1 por pixel muda de significado com a resolução. As métricas por comprimento e as relaxadas são mais comparáveis.
- **Comparação S2 degradado × sensor real.** A degradação por média ignora a PSF real, o ruído e a calibração do sensor de 64 m. O ponto do CBERS real mostra quanto disso importa.
- **Modo rápido.** AOIs de 9,6 km, 12 datas e poucas épocas: os ICs são largos, e os resultados são indicativos, não conclusivos.

**Próximos passos**
- Rodar o **modo completo** (AOIs de 28,8 km, 23 datas, todos os modelos em todos os experimentos, U-Net e Landsat 30 m real).
- Interpretar o **conjunto ouro** no QGIS e recalcular as métricas com ele como referência principal.
- Incluir o **Amazon Road Dataset (Imazon)** quando houver download público, e testar o rótulo de estrada como OSM ∪ Imazon.
- Adicionar o **Amazonia-1 WFI** (compondo cenas em cubo próprio) e o **CBERS-4A WPM 8 m** como ponto intermediário de resolução.
- Usar contexto espacial (texturas, U-Net/transformers espaço-temporais) em vez de só pixel, já que estradas são **estruturas lineares**.

### Status das fontes de dados nesta execução

In [ ]:
st = D.status_table()
print(f"tempo total do notebook: {(time.time() - T_INICIO) / 60:.1f} min")
st

## 13. Referências

- Botelho Jr., J.; Costa, S. C. P.; Ribeiro, J. G.; Souza Jr., C. M. (2022). *Mapping Roads in the Brazilian Amazon with Artificial Intelligence and Sentinel-2*. **Remote Sensing**, 14(15), 3625. https://doi.org/10.3390/rs14153625
- Ferreira, K. R. et al. (2020). *Earth Observation Data Cubes for Brazil: Requirements, Methodology and Products*. **Remote Sensing**, 12(24), 4033. (Brazil Data Cube)
- Souza, C. M. et al. (2020). *Reconstructing Three Decades of Land Use and Land Cover Changes in Brazilian Biomes with Landsat Archive and Earth Engine*. **Remote Sensing**, 12(17), 2735. (MapBiomas)
- MapBiomas: Coleção 10 da Série Anual de Mapas de Cobertura e Uso da Terra do Brasil; pontos de validação (LAPIG/TVI). https://brasil.mapbiomas.org
- Zanaga, D. et al. (2022). *ESA WorldCover 10 m 2021 v200*. https://doi.org/10.5281/zenodo.7254221
- Pekel, J.-F. et al. (2016). *High-resolution mapping of global surface water and its long-term changes*. **Nature**, 540, 418–422. (JRC GSW)
- Microsoft (2023). *Global ML Building Footprints*. https://github.com/microsoft/GlobalMLBuildingFootprints
- OpenStreetMap contributors (2026). Dados disponíveis sob ODbL. https://www.openstreetmap.org
- IBGE (2025). *Base Cartográfica Contínua do Brasil, 1:250 000 (BC250)*.
- DNIT. *Sistema Nacional de Viação (SNV)*.
- Pelletier, C.; Webb, G. I.; Petitjean, F. (2019). *Temporal Convolutional Neural Network for the Classification of Satellite Image Time Series*. **Remote Sensing**, 11(5), 523. (TempCNN)
- Garnot, V. S. F.; Landrieu, L. (2020). *Lightweight Temporal Self-Attention for Classifying Satellite Images Time Series*. AALTD/ECML-PKDD. (L-TAE)
- Lin, T.-Y. et al. (2017). *Focal Loss for Dense Object Detection*. ICCV.
- Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*. NeurIPS.
- Breiman, L. (2001). *Random Forests*. **Machine Learning**, 45, 5–32.
- Ronneberger, O.; Fischer, P.; Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI.
- Roberts, D. R. et al. (2017). *Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure*. **Ecography**, 40, 913–929. (validação em blocos)
- Wiedemann, C.; Heipke, C.; Mayer, H.; Jamet, O. (1998). *Empirical evaluation of automatically extracted road axes*. (completude/correção por comprimento)
- Mnih, V. (2013). *Machine Learning for Aerial Image Labeling*. PhD thesis, Univ. Toronto. (precisão/revocação relaxadas)
- Planet Labs / NICFI (2020–2025). *Planet & NICFI Basemaps for Tropical Forest Monitoring*.